# `extract.ipynb` - Extracción y limpieza de la fuente Legacy (SQLite, por chunks)

`extraer_y_limpiar_sqlite()` lee la tabla `economia_paises` (20.000 filas) en lotes de `CHUNK_SIZE` mediante `pd.read_sql_query(..., chunksize=...)` - así nunca se carga la tabla completa en memoria de golpe (requisito de **procesamiento eficiente**).

Por cada fila:

1. Se normaliza el nombre del país.
2. Si no es un país europeo válido -> **cuarentena** (`"Entidad territorial no perteneciente a la region europea"`).
3. Si el PIB es incoherente (`≤ 0`) -> **cuarentena** (`"Valor de PIB incoherente (PIB <= 0)"`).
4. En caso contrario, se convierte la superficie a km² y el PIB a EUR, y la fila pasa a la lista de **válidos**. Si la divisa no estuviera soportada, también se desvía a cuarentena en vez de romper el pipeline.

> Depende de `config.ipynb`, `logging_config.ipynb` (`logger`) y `transform.ipynb`. No requiere PostgreSQL - solo lee la SQLite local, así que la celda de prueba se puede ejecutar sin tener Docker levantado.

In [ ]:
import logging
import sqlite3

import pandas as pd

## Función principal

In [ ]:
def extraer_y_limpiar_sqlite():
    """
    Lee la SQLite Legacy en lotes (chunks) para optimizar memoria,
    normaliza cada fila y separa los registros validos de los que
    deben ir a cuarentena. Devuelve (df_validos, df_cuarentena, total_leidos).
    """
    logger.info(f"Conectando a la base de datos Legacy SQLite: {SQLITE_PATH}")
    conn_sqlite = sqlite3.connect(SQLITE_PATH)

    filas_validas = []
    filas_cuarentena = []
    total_leidos = 0

    query = f"SELECT * FROM {SQLITE_TABLA_ORIGEN}"
    logger.info(f"Iniciando lectura por chunks (chunksize={CHUNK_SIZE}) de la tabla '{SQLITE_TABLA_ORIGEN}'")

    for numero_chunk, chunk in enumerate(pd.read_sql_query(query, conn_sqlite, chunksize=CHUNK_SIZE), start=1):
        total_leidos += len(chunk)
        for fila in chunk.itertuples(index=False):
            pais_original = fila.pais_nombre_local
            superficie_valor = fila.superficie_valor
            superficie_unidad = fila.superficie_unidad
            pib_valor = fila.pib_valor
            pib_divisa = fila.pib_divisa

            pais_normalizado = normalizar_nombre_pais(pais_original)

            # Regla de cuarentena 1: entidad no perteneciente a Europa
            if not es_pais_europeo_valido(pais_normalizado):
                filas_cuarentena.append((
                    pais_original, superficie_valor, superficie_unidad,
                    pib_valor, pib_divisa,
                    "Entidad territorial no perteneciente a la region europea",
                ))
                continue

            # Regla de cuarentena 2: PIB incoherente (<= 0)
            if es_pib_incoherente(pib_valor):
                filas_cuarentena.append((
                    pais_original, superficie_valor, superficie_unidad,
                    pib_valor, pib_divisa,
                    "Valor de PIB incoherente (PIB <= 0)",
                ))
                continue

            try:
                superficie_km2 = convertir_superficie_a_km2(superficie_valor, superficie_unidad)
                pib_eur = convertir_pib_a_eur(pib_valor, pib_divisa)
            except ValueError as error:
                filas_cuarentena.append((
                    pais_original, superficie_valor, superficie_unidad,
                    pib_valor, pib_divisa, f"Error de transformacion: {error}",
                ))
                continue

            filas_validas.append((pais_normalizado, superficie_km2, pib_eur))

        logger.info(f"Chunk #{numero_chunk} procesado ({len(chunk)} filas). Acumulado leido: {total_leidos}")

    conn_sqlite.close()

    df_validos = pd.DataFrame(filas_validas, columns=["nombre_pais", "superficie_km2", "pib_total_eur"])
    df_cuarentena = pd.DataFrame(filas_cuarentena, columns=[
        "pais_original", "superficie_valor_orig", "superficie_unidad_orig",
        "pib_valor_orig", "pib_divisa_orig", "motivo_rechazo",
    ])

    logger.info(
        f"Extraccion SQLite finalizada. Total leido: {total_leidos} | "
        f"Validos: {len(df_validos)} | Cuarentena: {len(df_cuarentena)}"
    )
    return df_validos, df_cuarentena, total_leidos

## Prueba rápida

Ejecuta la extracción completa contra la SQLite real (no escribe nada en PostgreSQL) y muestra el resultado: cuántas filas se leyeron, cuántas pasaron validación y cuántas fueron a cuarentena.

In [ ]:
df_validos_prueba, df_cuarentena_prueba, total_prueba = extraer_y_limpiar_sqlite()

print(f"Total leido: {total_prueba}")
print(f"Validos: {len(df_validos_prueba)}")
print(f"Cuarentena: {len(df_cuarentena_prueba)}")
print("\nPrimeras filas validas:")
print(df_validos_prueba.head(3).to_string(index=False))